In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)
n = 20000

pre_revenue = np.random.lognormal(mean=4, sigma=0.8, size=n)
treatment = np.random.binomial(1, 0.5, n)
noise = np.random.normal(0, 30, n)

post_revenue = 0.8*pre_revenue + 5*treatment + noise
df = pd.DataFrame({"treatment": treatment, "pre_28_revenue": pre_revenue, "post_28_revenue": post_revenue})
df.head()

,treatment,pre_28_revenue,post_28_revenue
0,0,81.237042,28.613210
1,1,48.880990,37.810441
2,1,91.665935,117.472447
3,1,184.642936,167.917421
4,1,45.271482,4.042328


In [4]:
X = df["pre_28_revenue"]
Y = df["post_28_revenue"]

theta = np.cov(X, Y, ddof=1)[0,1]/np.var(X, ddof=1)
df["post_28_revenue_cuped"] = Y - theta * (X - X.mean())
df.head()

,treatment,pre_28_revenue,post_28_revenue,post_28_revenue_cuped
0,0,81.237042,28.613210,24.171238
1,1,48.880990,37.810441,59.246144
2,1,91.665935,117.472447,104.689670
3,1,184.642936,167.917421,80.773636
4,1,45.271482,4.042328,28.364837


In [13]:
raw_effect = df.loc[df["treatment"]==1, "post_28_revenue"].mean() - df.loc[df["treatment"]==0, "post_28_revenue"].mean()
cuped_effect = df.loc[df["treatment"]==1, "post_28_revenue_cuped"].mean() - df.loc[df["treatment"]==0, "post_28_revenue_cuped"].mean()
raw_var = df["post_28_revenue"].var()
cuped_var = df["post_28_revenue_cuped"].var()
variance_reduction = 1 - cuped_var / raw_var
correlation = df[["pre_28_revenue", "post_28_revenue"]].corr()

If a user had unusually high pre-period revenue, CUPED expects part of their high post-period revenue to come from that baseline behavior and subtracts that predictable component. If a user had unusually low pre-period revenue, it adjusts upward.

So CUPED is essentially removing baseline-driven heterogeneity from the outcome. It does not try to change the treatment effect; it tries to reduce noise so the Treatment vs. Control comparison becomes more precise.

CUPED removes explainable noise, reduces variance and standard error, and therefore increases statistical power.

In [14]:
print("Raw effect:", raw_effect)
print("CUPED effect:", cuped_effect)
print("Raw variance:", raw_var)
print("CUPED variance:", cuped_var)
print("Variance reduction:", variance_reduction)
print("Pre vs. Post revenue correlation:", correlation)

Raw effect: 4.454206360889486
CUPED effect: 4.756700319700094
Raw variance: 4295.0383160535
CUPED variance: 902.7834320848823
Variance reduction: 0.7898078280907153
Pre vs. Post revenue correlation:                  pre_28_revenue  post_28_revenue
pre_28_revenue         1.000000         0.888711
post_28_revenue        0.888711         1.000000
